# IPL Data Exploratory Analysis

This notebook explores the Kaggle IPL dataset (2008-2024) with visualizations and insights for ML modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Load data
matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')

print(f"Matches: {len(matches)} records")
print(f"Deliveries: {len(deliveries)} records")
print("\nMatches Columns:")
print(matches.columns.tolist())
print("\nDeliveries Columns:")
print(deliveries.columns.tolist())

## Dataset Overview

In [ ]:
# Display first few records
print("First 3 matches:")
print(matches[['id', 'season', 'date', 'team1', 'team2', 'venue', 'winner']].head(3))

print("\nFirst 5 deliveries:")
print(deliveries.head(5))

# Check for missing values
print("\nMissing values in matches:")
print(matches.isnull().sum())

print("\nMissing values in deliveries:")
print(deliveries.isnull().sum())

## Season Analysis

In [ ]:
# Matches per season
matches_per_season = matches.groupby('season').size()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
ax[0].bar(matches_per_season.index, matches_per_season.values, color='steelblue')
ax[0].set_xlabel('Season')
ax[0].set_ylabel('Number of Matches')
ax[0].set_title('Matches per Season')
ax[0].grid(axis='y')

# Season statistics
season_stats = matches.groupby('season').agg({
    'id': 'count',
    'venue': 'nunique'
}).rename(columns={'id': 'matches', 'venue': 'venues'})

print("\nMatches and Venues per Season:")
print(season_stats)

ax[1].plot(season_stats.index, season_stats['matches'], marker='o', linewidth=2, color='darkred')
ax[1].fill_between(season_stats.index, season_stats['matches'], alpha=0.3, color='red')
ax[1].set_xlabel('Season')
ax[1].set_ylabel('Number of Matches')
ax[1].set_title('Match Trend Over Seasons')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Team Analysis

In [ ]:
# All teams (combining team1 and team2)
all_teams = pd.concat([matches['team1'], matches['team2']])
team_counts = all_teams.value_counts()

print(f"Total unique teams: {len(team_counts)}")
print("\nMatches played per team:")
print(team_counts.head(10))

# Win rate by team
team_wins = matches['winner'].value_counts()
win_rate = (team_wins / team_counts * 100).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Top teams by matches
top_teams = team_counts.head(10)
ax[0].barh(range(len(top_teams)), top_teams.values, color='teal')
ax[0].set_yticks(range(len(top_teams)))
ax[0].set_yticklabels(top_teams.index)
ax[0].set_xlabel('Number of Matches')
ax[0].set_title('Top 10 Teams by Matches Played')
ax[0].invert_yaxis()

# Win rates
top_win_rates = win_rate[win_rate.index.isin(team_counts.head(10).index)].sort_values(ascending=False)
colors = ['green' if x > 50 else 'orange' for x in top_win_rates.values]
ax[1].barh(range(len(top_win_rates)), top_win_rates.values, color=colors)
ax[1].set_yticks(range(len(top_win_rates)))
ax[1].set_yticklabels(top_win_rates.index)
ax[1].set_xlabel('Win Rate (%)')
ax[1].set_title('Win Rate by Team (Top 10)')
ax[1].invert_yaxis()
ax[1].axvline(50, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Venue Analysis

In [ ]:
# Matches per venue
venue_counts = matches['venue'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(venue_counts)), venue_counts.values, color='coral')
ax.set_xticks(range(len(venue_counts)))
ax.set_xticklabels(venue_counts.index, rotation=45, ha='right')
ax.set_ylabel('Number of Matches')
ax.set_title('Top 10 Venues by Match Count')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal unique venues: {matches['venue'].nunique()}")
print("\nTop 10 venues:")
print(venue_counts)

## Toss Analysis

In [ ]:
# Toss decision distribution
toss_decision = matches['toss_decision'].value_counts()
print("Toss Decision Distribution:")
print(toss_decision)

# Toss winner vs match winner
toss_win_correlation = matches[matches['toss_winner'] == matches['winner']].shape[0] / len(matches) * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Toss decision pie
ax[0].pie(toss_decision.values, labels=toss_decision.index, autopct='%1.1f%%', colors=['#ff9999', '#66b3ff'])
ax[0].set_title('Toss Decision Distribution')

# Toss winner = Match winner rate
labels = ['Toss Winner\nAlso Won Match', 'Toss Winner\nLost Match']
values = [toss_win_correlation, 100 - toss_win_correlation]
ax[1].pie(values, labels=labels, autopct='%1.1f%%', colors=['#90EE90', '#FFB6C1'])
ax[1].set_title('Correlation: Toss Win vs Match Win')

plt.tight_layout()
plt.show()

print(f"\nToss winner won the match: {toss_win_correlation:.2f}% of the time")

## Deliveries Analysis

In [ ]:
## Innings Score Distribution

In [ ]:
# Aggregate innings scores
innings_scores = deliveries.groupby(['match_id', 'inning', 'batting_team']).agg({
    'runs': 'sum',
    'is_wicket': 'sum'
}).reset_index()
innings_scores.columns = ['match_id', 'inning', 'team', 'total_runs', 'wickets']

print(f"\nInnings Statistics:")
print(f"Average runs per innings: {innings_scores['total_runs'].mean():.2f}")
print(f"Median runs per innings: {innings_scores['total_runs'].median():.2f}")
print(f"Max runs in an innings: {innings_scores['total_runs'].max()}")
print(f"Min runs in an innings: {innings_scores['total_runs'].min()}")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
ax[0].hist(innings_scores['total_runs'], bins=30, color='mediumpurple', edgecolor='black')
ax[0].set_xlabel('Total Runs')
ax[0].set_ylabel('Frequency')
ax[0].set_title('Distribution of Innings Scores')
ax[0].axvline(innings_scores['total_runs'].mean(), color='red', linestyle='--', label='Mean')
ax[0].legend()
ax[0].grid(axis='y', alpha=0.3)

# Runs vs Wickets
ax[1].scatter(innings_scores['wickets'], innings_scores['total_runs'], alpha=0.5, s=30, color='teal')
ax[1].set_xlabel('Wickets Lost')
ax[1].set_ylabel('Total Runs')
ax[1].set_title('Innings Runs vs Wickets Lost')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Insights Summary

In [ ]:
print("\n" + "="*60)
print("KEY INSIGHTS FOR ML MODELING")
print("="*60)

print(f"\n1. DATASET SIZE:")
print(f"   - Total matches: {len(matches)}")
print(f"   - Total deliveries: {len(deliveries)}")
print(f"   - Time span: {matches['date'].min()} to {matches['date'].max()}")

print(f"\n2. TEAMS:")
print(f"   - Unique teams: {matches['team1'].nunique() + matches['team2'].nunique()}")
print(f"   - Most active: {team_counts.index[0]} ({team_counts.values[0]} matches)")

print(f"\n3. VENUES:")
print(f"   - Unique venues: {matches['venue'].nunique()}")
print(f"   - Most matches: {venue_counts.index[0]} ({venue_counts.values[0]} matches)")

print(f"\n4. SCORES:")
print(f"   - Average runs/innings: {innings_scores['total_runs'].mean():.0f}")
print(f"   - Average wickets/innings: {innings_scores['wickets'].mean():.2f}")

print(f"\n5. TOSS IMPACT:")
print(f"   - Toss winner win rate: {toss_win_correlation:.1f}%")
print(f"   - Suggests moderate importance for modeling")

print(f"\n6. DATA QUALITY:")
print(f"   - Missing winner values: {matches['winner'].isna().sum()}")
print(f"   - Missing toss_decision: {matches['toss_decision'].isna().sum()}")
print(f"   - Should handle appropriately in preprocessing")

print("\n" + "="*60)